# Conduct a analysis of the performance


In [ ]:
import pandas
import duckdb
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# 1.Read & load the results from the databases crash test !

In [3]:
file_path = '/Users/macbook/Development/database_crash_test/benchmarker/database_benchmark_results.csv'
results_df = pandas.read_csv(file_path)
results_df.columns

Index(['query', 'original_query', 'database_type', 'execution_time_ms',
       'cpu_usage_percent', 'memory_usage_mb', 'memory_usage_percent',
       'disk_read_mb', 'disk_write_mb', 'network_in_mb', 'network_out_mb',
       'result_rows', 'result_size_mb', 'failed'],
      dtype='object')

In [4]:
# check if results return failed queries
failed = results_df[results_df['failed'] == True]
failed

#print(failed['original_query'].values)

,query,original_query,database_type,execution_time_ms,cpu_usage_percent,memory_usage_mb,memory_usage_percent,disk_read_mb,disk_write_mb,network_in_mb,network_out_mb,result_rows,result_size_mb,failed
42,/* The INTERVAL creates an error in mysql */ /...,-- The INTERVAL creates an error in mysql\n\n-...,ClickHouseHandler,61.998129,6.2551,836.117188,20.413017,0.0,0.0,0.002766,0.004555,0,0.0,True


In [6]:
# Display initial data info
results_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   query                 44 non-null     object 
 1   original_query        44 non-null     object 
 2   database_type         44 non-null     object 
 3   execution_time_ms     44 non-null     float64
 4   cpu_usage_percent     44 non-null     float64
 5   memory_usage_mb       44 non-null     float64
 6   memory_usage_percent  44 non-null     float64
 7   disk_read_mb          44 non-null     float64
 8   disk_write_mb         44 non-null     float64
 9   network_in_mb         44 non-null     float64
 10  network_out_mb        44 non-null     float64
 11  result_rows           44 non-null     int64  
 12  result_size_mb        44 non-null     float64
 13  failed                44 non-null     bool   
dtypes: bool(1), float64(9), int64(1), object(3)
memory usage: 4.6+ KB


In [8]:
results_df.describe()

,execution_time_ms,cpu_usage_percent,memory_usage_mb,memory_usage_percent,disk_read_mb,disk_write_mb,network_in_mb,network_out_mb,result_rows,result_size_mb
count,44.000000,44.000000,44.000000,44.000000,44.0,44.0,44.000000,44.000000,44.000000,44.000000
mean,280.579908,3.548964,469.718306,15.125782,0.0,0.0,0.008700,1.361147,42816.818182,1.236386
std,424.818061,12.430645,291.635162,3.919215,0.0,0.0,0.018827,2.849215,72302.088789,2.008356
min,3.624916,0.002186,91.125000,8.898926,0.0,0.0,0.000000,0.000000,0.000000,0.000000
25%,38.366973,0.035220,172.292969,12.556839,0.0,0.0,0.001117,0.001268,1.000000,0.000149
50%,125.228047,0.599989,459.949219,15.779781,0.0,0.0,0.002076,0.004548,100.000000,0.006992
75%,214.842975,3.811686,749.439453,18.296862,0.0,0.0,0.003363,0.369423,48204.000000,1.404179
max,1680.254936,82.681051,910.074219,22.218609,0.0,0.0,0.077224,9.390633,166536.000000,5.082401


# 2. Analyse the results with SQL in duckdb

In [9]:
results_db = duckdb.sql("SELECT * FROM results_df")
results_db

┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────┬────────────────────┬────────────────────┬─────────────────┬──────────────────────┬──────────────┬───────────────┬────────────────────┬────────────────────┬─────────────┬────────────────────┬─────────┐
│                             

In [10]:
results_df.columns 

Index(['query', 'original_query', 'database_type', 'execution_time_ms',
       'cpu_usage_percent', 'memory_usage_mb', 'memory_usage_percent',
       'disk_read_mb', 'disk_write_mb', 'network_in_mb', 'network_out_mb',
       'result_rows', 'result_size_mb', 'failed'],
      dtype='object')

## Overall Performance Analysis

### Execution time

In [37]:
# get average, median, min, max execution time
execution_time = duckdb.sql(
    """SELECT 
    database_type,
    ROUND(AVG(execution_time_ms), 2) as avg_execution_time_ms,
    ROUND(MEDIAN(execution_time_ms), 2) as median_execution_time_ms,
    ROUND(MIN(execution_time_ms), 2) as min_execution_time_ms, 
    ROUND(MAX(execution_time_ms), 2) as max_execution_time_ms,
    FROM results_df
    GROUP BY database_type
    ORDER BY avg_execution_time_ms
    """
    )
execution_time

┌───────────────────┬───────────────────────┬──────────────────────────┬───────────────────────┬───────────────────────┐
│   database_type   │ avg_execution_time_ms │ median_execution_time_ms │ min_execution_time_ms │ max_execution_time_ms │
│      varchar      │        double         │          double          │        double         │        double         │
├───────────────────┼───────────────────────┼──────────────────────────┼───────────────────────┼───────────────────────┤
│ DuckDBHandler     │                 71.17 │                    13.28 │                  3.62 │                242.34 │
│ ClickHouseHandler │                276.81 │                    63.32 │                 31.23 │               1229.81 │
│ PostgresHandler   │                350.82 │                   160.51 │                119.89 │               1680.25 │
│ MySQLHandler      │                423.52 │                   160.93 │                  33.8 │               1419.38 │
└───────────────────┴───────────

In [ ]:
"""
# grouped median/avg execution time per database
databases = results_df['database_type'].unique().tolist()

avg_median_perf = go.Figure(
    data=[
    go.Bar(name='Average', x=databases, y=avg_execution_time['avg_execution_time_ms']),
    go.Bar(name='Median', x=databases, y=avg_execution_time['median_execution_time_ms'])
])
avg_median_perf.update_layout(
    barmode='group', 
    template=template, 
    title='Average vs Median Execution Time by Database Type')
avg_median_perf.show()
"""

In [40]:
# define dark theme
template = 'plotly_dark'

# create a ploty viz 
median_execution_time_plot = px.bar(
    execution_time, 
    x='database_type', 
    y='median_execution_time_ms',
    title='Median Execution Time in ms (lower is better)',
    template=template,
    #barmode='group',
)

median_execution_time_plot.update_traces(
    marker=
        {'color':'magenta'}
    
    )
median_execution_time_plot.show()

### Cpu usage

In [ ]:
# cpu usage in percent
cpu_usage = duckdb.sql(
    """SELECT 
    database_type,
    ROUND(AVG(cpu_usage_percent), 2) as avg_cpu_percent,
    ROUND(MEDIAN(cpu_usage_percent), 2) as median_cpu_usage_percent,
    ROUND(MIN(cpu_usage_percent), 2) as min_cpu_usage_percent, 
    ROUND(MAX(cpu_usage_percent), 2) as max_cpu_usage_percent,
    FROM results_df
    GROUP BY database_type
    ORDER BY avg_cpu_percent
    """
    )
cpu_usage

┌───────────────────┬─────────────────┬──────────────────────────┬───────────────────────┬───────────────────────┐
│   database_type   │ avg_cpu_percent │ median_cpu_usage_percent │ min_cpu_usage_percent │ max_cpu_usage_percent │
│      varchar      │     double      │          double          │        double         │        double         │
├───────────────────┼─────────────────┼──────────────────────────┼───────────────────────┼───────────────────────┤
│ DuckDBHandler     │            0.01 │                     0.01 │                   0.0 │                  0.01 │
│ PostgresHandler   │            0.08 │                     0.07 │                  0.04 │                  0.14 │
│ MySQLHandler      │            1.77 │                     1.32 │                  1.06 │                   5.2 │
│ ClickHouseHandler │           12.34 │                     4.54 │                  3.76 │                 82.68 │
└───────────────────┴─────────────────┴──────────────────────────┴──────────────

In [36]:
# create a ploty viz 
cpu_usage_plot = px.bar(
    cpu_usage, 
    x='database_type', 
    y='median_cpu_usage_percent',
    title='Median CPU usage in % (lower is better)',
    template=template,
    #barmode='group',
)

cpu_usage_plot.update_traces(
    marker=
        {'color':'darkblue'}
    
    )
cpu_usage_plot.show()

### Memory usage

In [41]:
mem_usage = duckdb.sql(
    """SELECT 
    database_type,
    ROUND(AVG(memory_usage_percent), 2) as avg_memory_usage_percent,
    ROUND(MEDIAN(memory_usage_percent), 2) as median_memory_usage_percent,
    ROUND(MIN(memory_usage_percent), 2) as min_memory_usage_percent, 
    ROUND(MAX(memory_usage_percent), 2) as max_memory_usage_percent,
    FROM results_df
    GROUP BY database_type
    ORDER BY avg_memory_usage_percent
    """
    )

mem_usage

┌───────────────────┬──────────────────────────┬─────────────────────────────┬──────────────────────────┬──────────────────────────┐
│   database_type   │ avg_memory_usage_percent │ median_memory_usage_percent │ min_memory_usage_percent │ max_memory_usage_percent │
│      varchar      │          double          │           double            │          double          │          double          │
├───────────────────┼──────────────────────────┼─────────────────────────────┼──────────────────────────┼──────────────────────────┤
│ DuckDBHandler     │                    10.23 │                         8.9 │                      8.9 │                     12.6 │
│ PostgresHandler   │                    13.91 │                       14.22 │                     9.12 │                     17.7 │
│ MySQLHandler      │                    17.53 │                       18.41 │                    13.61 │                    18.65 │
│ ClickHouseHandler │                    18.83 │                     

In [42]:
# create a ploty viz 
mem_usage_usage_plot = px.bar(
    mem_usage, 
    x='database_type', 
    y='median_memory_usage_percent',
    title='Median RAM usage in % (lower is better)',
    template=template,
    #barmode='group',
)

cpu_usage_plot.update_traces(
    marker=
        {'color':'darkred'}
    
    )
cpu_usage_plot.show()

In [51]:
cpu_execution = px.scatter(
   results_df,
   x='memory_usage_mb',
   y='execution_time_ms',
   color='database_type',
   title='Memory usage vs execution time',
   template=template
)
cpu_execution.show()

## Query execution time analysis

In [19]:
results_df.columns

Index(['query', 'original_query', 'database_type', 'execution_time_ms',
       'cpu_usage_percent', 'memory_usage_mb', 'memory_usage_percent',
       'disk_read_mb', 'disk_write_mb', 'network_in_mb', 'network_out_mb',
       'result_rows', 'result_size_mb', 'failed'],
      dtype='object')

In [ ]:
query_time = duckdb.sql(
    """
    SELECT 
        original_query as query,
        execution_time_ms
    FROM results_df
    """
)
